In [2]:
import re, random, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, Embedding, LSTM, Dense,
                                     Dropout, BatchNormalization, Bidirectional)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

# ── CONFIG ───────────────────────────────────────────────
DATA_PATH       = r'C:\Users\sandy\Downloads\Neural-Project-main\Neural-Project-main\final_neural_network_data.csv'
TREATMENT_PATH  = r'C:\Users\sandy\Downloads\Neural-Project-main\Neural-Project-main\skin_disease_dataset2222\treatments.csv'
MODEL_SAVE_PATH = 'best_disease_model.keras'

MAX_WORDS   = 5000
MAX_LEN     = 100
EMBED_DIM   = 64
LSTM_UNITS  = 128
DROPOUT     = 0.4
SEED        = 42

EPOCHS      = 15
BATCH_SIZE  = 8       # small batch suits tiny dataset
LR          = 0.001
N_FOLDS     = 5       # K-Fold
AUG_FACTOR  = 5       # how many augmented copies per sample

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(' Imports & config done.')

 Imports & config done.


In [3]:
df = pd.read_csv(DATA_PATH).dropna(subset=['Disease', 'Description'])
treatments_df = pd.read_csv(TREATMENT_PATH, usecols=['Disease', 'Treatment', 'Avoid']).dropna()

df['Disease']            = df['Disease'].str.strip().str.lower()
treatments_df['Disease'] = treatments_df['Disease'].str.strip().str.lower()

print(f'Total samples  : {len(df)}')
print(f'  Diseases        : {df["Disease"].nunique()}')
print()
print(df['Disease'].value_counts())

# ── Text cleaning ─────────────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))

def preprocess(text):
    text   = str(text).lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['Description'].apply(preprocess)

# ── Encode labels ─────────────────────────────────────────
le = LabelEncoder()
df['label'] = le.fit_transform(df['Disease'])
NUM_CLASSES  = len(le.classes_)
print(f'\nClasses ({NUM_CLASSES}): {list(le.classes_)}')

# ── Treatment map ─────────────────────────────────────────
treatment_map = {
    row['Disease']: {'treatment': row['Treatment'], 'avoid': row['Avoid']}
    for _, row in treatments_df.iterrows()
}

print('\n Data loaded and preprocessed.')

Total samples  : 103
  Diseases        : 8

Disease
eczema                13
warts                 13
impetigo              13
acne                  13
contact dermatitis    13
skin cancer           13
psoriasis             13
vitiligo              12
Name: count, dtype: int64

Classes (8): ['acne', 'contact dermatitis', 'eczema', 'impetigo', 'psoriasis', 'skin cancer', 'vitiligo', 'warts']

 Data loaded and preprocessed.


In [4]:
def augment_text(text, n=AUG_FACTOR):
    """Create n augmented versions by randomly dropping ~20% of words."""
    words   = text.split()
    results = []
    for _ in range(n):
        kept = [w for w in words if random.random() > 0.20]
        if len(kept) < 3:
            kept = words          # fallback: keep original
        results.append(' '.join(kept))
    return results

aug_texts  = []
aug_labels = []

for _, row in df.iterrows():
    for aug in augment_text(row['clean_text']):
        aug_texts.append(aug)
        aug_labels.append(row['label'])

aug_df = pd.DataFrame({'clean_text': aug_texts, 'label': aug_labels})

# Combine original + augmented
full_df = pd.concat([
    df[['clean_text', 'label']],
    aug_df
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

X_all = full_df['clean_text'].values
y_all = full_df['label'].values

print(f'Dataset after augmentation: {len(full_df)} samples')
print(' Augmentation done.')

Dataset after augmentation: 618 samples
 Augmentation done.


In [5]:
# Fit tokenizer on ALL text (original) so vocab is consistent
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_text'].values)

def encode(texts):
    return pad_sequences(tokenizer.texts_to_sequences(texts), maxlen=MAX_LEN)

X_enc = encode(X_all)
y_cat = to_categorical(y_all, NUM_CLASSES)

print(f' Encoded shape: {X_enc.shape}')

 Encoded shape: (618, 100)


In [6]:
def build_model(lr=LR):
    inp = Input(shape=(MAX_LEN,))
    x   = Embedding(MAX_WORDS, EMBED_DIM)(inp)
    x   = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
    x   = Dropout(DROPOUT)(x)
    x   = Bidirectional(LSTM(LSTM_UNITS // 2))(x)
    x   = BatchNormalization()(x)
    x   = Dense(64, activation='relu')(x)
    x   = Dropout(DROPOUT)(x)
    out = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

build_model().summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 100, 256)       │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           520 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 691,272 (2.64 MB)

 Trainable params: 691,016 (2.64 MB)

 Non-trainable params: 256 (1.00 KB)

In [7]:
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_scores  = []
best_val_acc = 0.0

for fold, (train_idx, val_idx) in enumerate(skf.split(X_enc, y_all), 1):
    print(f'\n{'='*55}')
    print(f'  Fold {fold} / {N_FOLDS}')
    print(f'{'='*55}')

    X_tr, X_val = X_enc[train_idx], X_enc[val_idx]
    y_tr, y_val = y_cat[train_idx], y_cat[val_idx]

    model = build_model()

    fold_path = f'fold_{fold}_model.keras'

    callbacks = [
        ModelCheckpoint(
            fold_path,
            save_best_only=True,
            monitor='val_accuracy',
            mode='max',
            verbose=0
        ),
        EarlyStopping(
            patience=15,
            monitor='val_accuracy',
            mode='max',
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=7,
            min_lr=1e-6,
            verbose=0
        )
    ]

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate this fold's best model
    fold_model   = load_model(fold_path)
    _, fold_acc  = fold_model.evaluate(X_val, y_val, verbose=0)
    fold_scores.append(fold_acc)
    print(f'\n Fold {fold} val_accuracy: {fold_acc*100:.2f}%')

    # Save the overall best model
    if fold_acc > best_val_acc:
        best_val_acc = fold_acc
        fold_model.save(MODEL_SAVE_PATH)
        print(f' New best model saved! ({best_val_acc*100:.2f}%)')

print(f'\n{'='*55}')
print(f'K-Fold Results:')
for i, s in enumerate(fold_scores, 1):
    print(f'  Fold {i}: {s*100:.2f}%')
print(f'  Mean : {np.mean(fold_scores)*100:.2f}% ± {np.std(fold_scores)*100:.2f}%')
print(f'  Best : {best_val_acc*100:.2f}%')
print(f'{'='*55}')
print(f'\n Best model saved to: {MODEL_SAVE_PATH}')


  Fold 1 / 5
Epoch 1/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.4109 - loss: 1.5991 - val_accuracy: 0.3710 - val_loss: 1.8935 - learning_rate: 0.0010
Epoch 2/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - accuracy: 0.8866 - loss: 0.4359 - val_accuracy: 1.0000 - val_loss: 1.3375 - learning_rate: 0.0010
Epoch 3/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9696 - loss: 0.1449 - val_accuracy: 1.0000 - val_loss: 0.8670 - learning_rate: 0.0010
Epoch 4/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.9858 - loss: 0.0928 - val_accuracy: 1.0000 - val_loss: 0.4309 - learning_rate: 0.0010
Epoch 5/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9919 - loss: 0.0521 - val_accuracy: 1.0000 - val_loss: 0.1412 - learning_rate: 0.0010
Epoch 6/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9838 - loss: 0.0606 - val_accuracy: 0.9919 - val_loss: 0.0971 - learning_rate: 0.0010
Epoch 7/15
62/62 ━━━━━━━━━━━━━━━━━━━━ 2s 28ms/step - accuracy: 0.9656 - loss: 0.

In [8]:
best_model = load_model(MODEL_SAVE_PATH)

# Evaluate on the original (non-augmented) data
X_orig = encode(df['clean_text'].values)
y_orig = df['label'].values

y_pred = np.argmax(best_model.predict(X_orig, verbose=0), axis=1)
acc    = accuracy_score(y_orig, y_pred)

print(f'Accuracy on original data : {acc*100:.2f}%\n')
print(classification_report(y_orig, y_pred, target_names=le.classes_))

Accuracy on original data : 100.00%

                    precision    recall  f1-score   support

              acne       1.00      1.00      1.00        13
contact dermatitis       1.00      1.00      1.00        13
            eczema       1.00      1.00      1.00        13
          impetigo       1.00      1.00      1.00        13
         psoriasis       1.00      1.00      1.00        13
       skin cancer       1.00      1.00      1.00        13
          vitiligo       1.00      1.00      1.00        12
             warts       1.00      1.00      1.00        13

          accuracy                           1.00       103
         macro avg       1.00      1.00      1.00       103
      weighted avg       1.00      1.00      1.00       103



In [9]:
# Make sure best_model is loaded
try:
    best_model
except NameError:
    best_model = load_model(MODEL_SAVE_PATH)
    print('Model loaded.')

def predict_disease(text, top_k=3):
    clean   = preprocess(text)
    encoded = encode([clean])
    probs   = best_model.predict(encoded, verbose=0)[0]

    top_idx    = int(np.argmax(probs))
    disease    = le.classes_[top_idx]
    confidence = probs[top_idx] * 100

    top_k_idx = np.argsort(probs)[::-1][:top_k]
    top_preds = [
        f"{le.classes_[i].title()} ({probs[i]*100:.1f}%)"
        for i in top_k_idx
    ]

    info = treatment_map.get(disease, {})
    return {
        'disease'    : disease.title(),
        'confidence' : f'{confidence:.1f}%',
        'treatment'  : info.get('treatment', 'Consult a doctor'),
        'avoid'      : info.get('avoid', 'N/A'),
        'top_k'      : top_preds
    }


# ── Interactive input loop ────────────────────────────────
print('=' * 58)
print('   Disease Prediction from Symptom Description')
print('=' * 58)
print('Describe the skin symptoms in English.')
print("Type 'exit' to quit.\n")

while True:
    user_input = input(' Enter symptom description: ').strip()

    if not user_input or user_input.lower() in ('exit', 'quit'):
        print(' Goodbye!')
        break

    result = predict_disease(user_input)

    print('\n' + '-' * 58)
    print(f"  Predicted Disease : {result['disease']}")
    print(f"  Confidence        : {result['confidence']}")
    print(f"  Treatment         : {result['treatment']}")
    print(f"   Avoid             : {result['avoid']}")
    print(f"  Top-{len(result['top_k'])}            : {' | '.join(result['top_k'])}")
    print('-' * 58 + '\n')

   Disease Prediction from Symptom Description
Describe the skin symptoms in English.
Type 'exit' to quit.


----------------------------------------------------------
  Predicted Disease : Eczema
  Confidence        : 31.3%
  Treatment         : Use moisturizers and topical corticosteroids; avoid irritants
   Avoid             : Avoid harsh soaps and scratching
  Top-3            : Eczema (31.3%) | Acne (14.5%) | Contact Dermatitis (12.5%)
----------------------------------------------------------


----------------------------------------------------------
  Predicted Disease : Vitiligo
  Confidence        : 21.6%
  Treatment         : Topical steroids, phototherapy, or skin camouflage
   Avoid             : Avoid sunburn and skin trauma
  Top-3            : Vitiligo (21.6%) | Contact Dermatitis (15.5%) | Eczema (14.8%)
----------------------------------------------------------


----------------------------------------------------------
  Predicted Disease : Warts
  Confidence      